# Vaccine Sense — Aplicação 18

## Coleta e rotulagem do dataset

Traz do InfluxDB as medições da caixa térmica, você rotula cada rodada e salva
o dataset que a Aplicação 19 vai usar para treinar o modelo.

```
ESP32 → MQTT Broker (local) → Node-RED (local) → InfluxDB (cloud) → Colab
```

> Um exemplo de como o dataset final deve ficar está em
> `dataset_gerado/vaccinesense_dataset.csv`.


## 1. Pacotes


In [ ]:
!pip -q install influxdb3-python pandas pyarrow


## 2. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from influxdb_client_3 import InfluxDBClient3


## 3. Credenciais

Os mesmos valores que você configurou no nó InfluxDB do Node-RED.


In [ ]:
INFLUX_URL         = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN       = "XXXXX"
INFLUX_ORG         = "XXXXX"
INFLUX_BUCKET      = "XXXXX"
INFLUX_MEASUREMENT = "vaccinesense_raw_2026"

client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)


## 4. O que existe no bucket

Rode estas duas consultas **antes** da consulta grande. Nome de measurement
errado não dá erro: devolve resultado vazio.


In [ ]:
# SQL: lista todas as tabelas/measurements do bucket
tabelas = client.query("SHOW TABLES", language="sql")
display(tabelas.to_pandas())


In [ ]:
# Lista os campos do measurement do Vaccine Sense
colunas = client.query(f'SHOW COLUMNS FROM "{INFLUX_MEASUREMENT}"', language="sql")
display(colunas.to_pandas())


## 5. Trazer as medições

Ajuste o intervalo para cobrir toda a sua coleta.


In [ ]:
query = f"""
SELECT
  "time",
  "device",
  "rodada",
  "id",
  "tempInterna",
  "tempExterna",
  "umidade",
  "luz",
  "criticidade",
  "distancia",
  "tempoForaDaFaixa"
FROM "{INFLUX_MEASUREMENT}"
WHERE time >= now() - interval '6 hours'
ORDER BY time
"""

tabela = client.query(query=query, language="sql")
df = tabela.to_pandas()
df["time"] = pd.to_datetime(df["time"], utc=True)
df = (df.rename(columns={"time": "timestamp"})
        .set_index("timestamp")
        .sort_index())

print(f"{len(df)} medições")
df.tail(15)


## 6. Que rodadas chegaram

Cada vez que você apertou o botão para iniciar, o firmware abriu uma rodada
nova e a numerou. O protocolo pede **24 rodadas**, com cerca de 40 medições
cada.

Se você reiniciou o ESP32 no meio da coleta, a numeração recomeça do 1 — olhe
as colunas de início e fim para perceber.


In [ ]:
resumo = df.groupby("rodada").agg(
    medicoes=("tempInterna", "size"),
    inicio=("tempInterna", lambda s: s.index.min().strftime("%H:%M:%S")),
    fim=("tempInterna", lambda s: s.index.max().strftime("%H:%M:%S")),
)
display(resumo)


## 7. Olhar os dados antes de rotular

Procure nos gráficos onde a tampa foi aberta: a luz sobe e a distância muda.
E onde a temperatura interna saiu da faixa aceita para a criticidade.


In [ ]:
fig, eixos = plt.subplots(5, 1, figsize=(14, 11), sharex=True)

series = [
    ("tempInterna", "Temp. interna (°C)"),
    ("tempExterna", "Temp. externa (°C)"),
    ("umidade",     "Umidade externa (%)"),
    ("luz",         "Luz na caixa"),
    ("distancia",   "Distância (cm)"),
]

for eixo, (coluna, titulo) in zip(eixos, series):
    eixo.plot(df.index, df[coluna], linewidth=1)
    eixo.set_ylabel(titulo)
    eixo.grid(alpha=0.3)

eixos[-1].set_xlabel("horário")
fig.suptitle("Medições coletadas")
plt.tight_layout()
plt.show()


## 8. Rotular

Aqui é o trabalho de especialista: **você sabe** o que fez com a caixa em cada
rodada, e é isso que vira o rótulo.

> O rótulo **não** vem de nenhum `if` do firmware nem de filtro por horário.
> Ele vem da rodada: o botão numerou, você anotou o que fez.
> O único `if` que existe no firmware acende o LED e não é publicado.

Cruze a tabela do passo 6 com a sua folha de anotação e preencha o dicionário.
Rodadas que não estiverem no dicionário ficam marcadas como `DESCARTAR`.

Confira no gráfico do passo 7 se bate: numa rodada de tampa aberta a luz sobe e
a distância muda. Se não bater, a anotação está trocada.


In [ ]:
# A chave é o NÚMERO da rodada, do jeito que o botão numerou.
# Este é o protocolo de 24 rodadas do README — ajuste se você mudou algo.
SITUACAO = {
    # --- não-perigo -------------------------------------------------
    "1":  "TRANSPORTE_OK",     # padrão, varrendo a faixa segura
    "2":  "AMBIENTE_HOSTIL",   # padrão preservada no calor
    "3":  "TRANSPORTE_OK",     # padrão perto de 8 °C
    "4":  "TRANSPORTE_OK",     # padrão perto de 8 °C
    "5":  "TRANSPORTE_OK",     # padrão perto de 2 °C
    "6":  "TRANSPORTE_OK",     # padrão perto de 2 °C
    "7":  "TRANSPORTE_OK",     # crítica entre 4 e 6 °C
    "8":  "TRANSPORTE_OK",     # crítica entre 4 e 6 °C
    "9":  "AMBIENTE_HOSTIL",   # padrão preservada no calor
    "10": "AMBIENTE_HOSTIL",   # crítica preservada no calor
    "11": "TRANSPORTE_OK",     # pouca carga, luz da sala variando
    "12": "TRANSPORTE_OK",     # pouca carga, luz da sala variando

    # --- perigo -----------------------------------------------------
    "13": "CARGA_EM_PERIGO",   # padrão superaquecida
    "14": "CARGA_EM_PERIGO",   # padrão superaquecida
    "15": "CARGA_EM_PERIGO",   # padrão congelando
    "16": "CARGA_EM_PERIGO",   # padrão congelando
    "17": "CARGA_EM_PERIGO",   # crítica acima de 6 °C
    "18": "CARGA_EM_PERIGO",   # crítica acima de 6 °C
    "19": "CARGA_EM_PERIGO",   # crítica abaixo de 4 °C
    "20": "CARGA_EM_PERIGO",   # crítica abaixo de 4 °C
    "21": "CARGA_EM_PERIGO",   # tampa aberta
    "22": "CARGA_EM_PERIGO",   # tampa aberta em ambiente hostil
    "23": "CARGA_EM_PERIGO",   # tampa entreaberta
    "24": "CARGA_EM_PERIGO",   # tampa entreaberta
}

df["situacao"] = df["rodada"].astype(str).map(SITUACAO).fillna("DESCARTAR")

display(df["situacao"].value_counts())


## 9. Descartar as transições

Os primeiros segundos de cada rodada ainda carregam o efeito do que veio antes:
o sensor estabilizando, o ar se misturando. Fora do dataset.

Este é o **único** descarte que fazemos. Não removemos medições por estarem
fora da faixa esperada: se o valor foi medido, ele é dado.


In [ ]:
SEGUNDOS_DESCARTE = 10

# a 1 Hz, a contagem dentro da rodada equivale a segundos
df["segundo_na_rodada"] = df.groupby("rodada").cumcount()

antes = len(df)
df = df[(df["situacao"] != "DESCARTAR") &
        (df["segundo_na_rodada"] >= SEGUNDOS_DESCARTE)].copy()
df = df.drop(columns=["segundo_na_rodada"])

print(f"{antes} → {len(df)} medições  ({antes - len(df)} descartadas)")


## 10. Conferir o resultado

As três situações precisam estar presentes. E o alvo binário da Aplicação 19
— `CARGA_EM_PERIGO` contra o resto — precisa ficar equilibrado.

Se `AMBIENTE_HOSTIL` ficou de fora, o modelo vai confundir dia quente com
tampa aberta.


In [ ]:
display(df["situacao"].value_counts())

perigo = (df["situacao"] == "CARGA_EM_PERIGO").sum()
print(f"\nAlvo binário:  perigo = {perigo}   não-perigo = {len(df) - perigo}")


In [ ]:
cores = {
    "TRANSPORTE_OK":   "tab:blue",
    "AMBIENTE_HOSTIL": "tab:orange",
    "CARGA_EM_PERIGO": "tab:red",
}

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

for situacao, grupo in df.groupby("situacao"):
    eixos[0].scatter(grupo["criticidade"], grupo["tempInterna"], s=8,
                     label=situacao, color=cores.get(situacao), alpha=0.6)
    eixos[1].scatter(grupo["distancia"], grupo["luz"], s=8,
                     label=situacao, color=cores.get(situacao), alpha=0.6)

# as faixas aceitas, para conferir a rotulagem
eixos[0].axvline(50, color="gray", linestyle="--", linewidth=1)
eixos[0].plot([0, 50], [2, 2], color="green", linewidth=1.5)
eixos[0].plot([0, 50], [8, 8], color="green", linewidth=1.5)
eixos[0].plot([50, 100], [4, 4], color="green", linewidth=1.5)
eixos[0].plot([50, 100], [6, 6], color="green", linewidth=1.5)
eixos[0].set_xlabel("criticidade")
eixos[0].set_ylabel("Temp. interna (°C)")
eixos[0].set_title("A faixa aceita muda com a criticidade")

eixos[1].set_xlabel("distância (cm)")
eixos[1].set_ylabel("luz")
eixos[1].set_title("Distância alta nem sempre é tampa aberta")

for eixo in eixos:
    eixo.grid(alpha=0.3)
    eixo.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.show()


No gráfico da esquerda, as linhas verdes são as faixas aceitas: 2 a 8 °C para
carga padrão e 4 a 6 °C para carga crítica. Pontos vermelhos **fora** das
linhas e pontos azuis **dentro** delas confirmam que a rotulagem está coerente.

No gráfico da direita, repare que há pontos azuis com distância alta — são as
rodadas com pouca carga e a tampa fechada. Distância grande, sozinha, não
significa tampa aberta.


## 11. Salvar o dataset


In [ ]:
ARQUIVO = "vaccinesense_dataset.csv"

df.to_csv(ARQUIVO)
print(f"{len(df)} linhas salvas em {ARQUIVO}")
df.head()


In [ ]:
from google.colab import files

files.download(ARQUIVO)


---

## Atenção para a Aplicação 19

Nem toda coluna do dataset é uma feature. Ao treinar o modelo, **três colunas
precisam sair**:

| Coluna | Por quê |
|---|---|
| `id` | contador sequencial: o modelo separaria as classes pela ordem de coleta, não pelos sensores |
| `timestamp` | a mesma coisa, em outra forma |
| `tempoForaDaFaixa` | é derivado da temperatura interna e só cresce dentro da rodada |

Também saem `device` e `rodada` — são metadados do experimento — e `situacao`,
que é o alvo.

As features são as **seis medições**: `tempInterna`, `tempExterna`, `umidade`,
`luz`, `criticidade`, `distancia`.

A coluna `rodada` sai das features, mas **guarde o CSV com ela**: a Aplicação 19
precisa dela para dividir treino e teste sem vazamento.
